#**≽^-⩊-^≼ Cats Cross Breeding - Generative AI**

Dieses Notebook trainiert ein Stable Diffusion 1.5 Modell auf Basis [eines Kaggle-Datensatzes mit 66 Katzenrassen](https://www.kaggle.com/datasets/nikolasgegenava/cat-breeds), um anschließend per Prompts Kreuzungen zu generieren.

-> [Emoji](https://emojicombos.com/cat)

## **Imports durchführen**

In [ ]:
!pip install -q kaggle
!pip install -q xformers bitsandbytes datasets

import os
import zipfile
from google.colab import files
from google.colab import drive

from PIL import Image
from collections import Counter
from torchvision.datasets import ImageFolder
import shutil
import random
from torch.utils.data import Dataset, random_split, DataLoader
from IPython.display import display, Image as IPImage
from pathlib import Path

import torch
import numpy as np
from diffusers import StableDiffusionPipeline, DDPMScheduler, UNet2DConditionModel, AutoencoderKL
from transformers import CLIPTextModel, CLIPTokenizer
from accelerate import Accelerator
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


## **Kaggle API Setup**

- Lädt kaggle.json hoch und richtet Kaggle API-Authentifizierung ein

In [ ]:
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"janaaa00","key":"7fb8f0015caa5d976e3012995bede7c2"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

## **Google Drive Mount**
- Google Drive anbinden, um später model & checkpoints hochzuladen

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **Kaggle Datensatz laden**
- Lädt den Datensatz runter, erstellt ein Verzeichnis, extrahiert darin die .zip-Datei und löscht diese
- Auslesen der Struktur, um zu sehen, ob alles geklappt hat + Ausgeben der Bilderanzahl für einen ersten Überblick

In [ ]:
!kaggle datasets download -d nikolasgegenava/cat-breeds

os.makedirs("data/cat_breeds", exist_ok=True)

with zipfile.ZipFile('cat-breeds.zip', 'r') as zip_ref:
    zip_ref.extractall('data/cat_breeds/')

os.remove("cat-breeds.zip")

Dataset URL: https://www.kaggle.com/datasets/nikolasgegenava/cat-breeds
License(s): MIT
  0% 0.00/40.3M [00:00<?, ?B/s]
100% 40.3M/40.3M [00:00<00:00, 1.56GB/s]


In [ ]:
for dirname, _, filenames in os.walk("data/cat_breeds"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

dataset = ImageFolder(root="data/cat_breeds/cat-breeds/cat-breeds")
print(f"number of images: {len(dataset)}")

data/cat_breeds/dataset_stats.csv
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0082.jpg
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0108.jpg
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0164.jpg
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0121.jpg
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0127.jpg
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0135.jpg
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0107.jpg
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0175.jpg
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0063.jpg
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0195.jpg
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0142.jpg
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0001.jpg
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0084.jpg
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0193.jpg
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0054.jpg
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0102.jpg
data/cat_breeds/cat-breeds/cat-breeds/ragdoll/0055.jpg
data/cat_breeds/cat-breeds/cat-

## **Daten bereinigen**
- Liest alle Dateien im Ordner cat_breeds aus
- Ziel: Schauen, welche Bildtypen vorhanden sind, damit im späteren Verlauf alle Bilddateien zu einem Typ umgewandelt werden können, da es besser für das Training ist (geringere Komplexität = stabileres Training).

**->** .jpg: 11276, .png: 8, entsprechend evtl. alle zu .jpg umwandeln.

In [ ]:
extensions = []

for root, _, files in os.walk("data/cat_breeds"):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        extensions.append(ext)

Counter(extensions)

Counter({'.csv': 1, '.jpg': 11276, '.png': 8})

### **Problem festgestellt**

- Datensatz soll teils nicht gut sein, s. https://www.kaggle.com/code/nunomvr/dataset-is-unusable-no-control-webscraping

**-> Deshalb:** Ordner runterladen und prüfen.

In [ ]:
shutil.make_archive('cat_breeds_dataset', 'zip', 'data/cat_breeds/cat-breeds/cat-breeds')
files.download('cat_breeds_dataset.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**-> Ergebnis:** Datenanalyse bestätigt Problem. Zudem sind die Bilder sehr klein (max. 200x200px), weshalb zu vermuten ist, dass spätere generierte Bilder verpixelt sein werden.

**-> Ziel jetzt:** Ordner, die nur "falsche" Bilder enthalten, werden gelöscht. Bei Ordnern, bei denen die Katzen halbwegs ok sind, werden nur falsche Bilder gelöscht.

**Also:**
- Datensatz als .zip-Datei herunterladen
- Lokal am PC jeden Ordner durchgehen und Bilder/Ordner, die "falsch" sind, manuell löschen
- Nach dem Bereinigen wieder hochladen -> cat_breeds_cleaned
- Danach prüfen, welche Datentypen vorhanden sind & Bilder zählen

**Löschvorgang:**
- Bilder, die zwar die Katze zeigen, aber auch ein Gesicht/Mensch (drinnen gelassen, wenn Mensch eher im Hintergrund war. Gelöscht, wenn Mensch mit Katze optisch konkurriert hat)
- Illustrationen von Katzen oder Bilder in Graustufen, um nur auf realen Bildern und gleichen Farbräumen zu trainieren, um einheitlich zu bleiben (verbessert Ergebnisse)
- Jegliche Bilder, die keine Katzen zeigen (z.B. nur Landschaft, Menschen, andere Tiere, Werbebilder, usw.)
- "vs. Bilder" (z.B. Maine Coon vs. Norwegian Forest Cat) wurden gelöscht, da dort zwei Katzenrassen gezeigt worden sind

**Eventuell problematisch:**
- Bilder mit Text/Beschriftung wurden drinnen gelassen, was jedoch dazu führen kann, dass das model später auch Bilder mit Text generiert. Das wurde gemacht, da teils ansonsten die Ordner sehr klein gewesen wären

**!** Durch das Aussortieren "falscher" Bilder kann sichergestellt werden, dass das Trainingsergebnis besser ist. Es ist jedoch zu testen, ob Katzenrassen, dessen Ordner durch vieles Löschen sehr klein geworden sind, nach dem Training overfitted sind bzw. nicht gut genug trainiert sind.


In [ ]:
files.upload()

with zipfile.ZipFile('cat_breeds_cleaned.zip', 'r') as zip_ref:
    zip_ref.extractall('data/cat_breeds_cleaned/')

os.remove("cat_breeds_cleaned.zip")

Saving cat_breeds_cleaned.zip to cat_breeds_cleaned.zip


In [ ]:
extensions = []

for root, _, files in os.walk("data/cat_breeds_cleaned"):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        extensions.append(ext)

Counter(extensions)

Counter({'.jpg': 8660})

**Nach manueller Bereinigung:** Von .jpg: 11276, .png: 8 zu .jpg: 8660

-> Keine pngs, muss also nichts konvertiert werden.

-> Aber: 2624 Bilder weniger.

## **Datensatz auf Training vorbereiten**
- Captions für jedes Bild erstellen (z.B. "a photo of a sphynx cat")
- Datensatz aufteilen -> Da es nur noch 8660 Bilder sind, wurde eine 70/15/15 Verteilung gewählt, um mehr Bilder für validation/test zu haben
- Ausgeben lassen, wie Verteilung pro Ordner ist und Gesamtverteilung anschauen (Prüfen, ob es so ok ist!)

In [ ]:
def create_captions_for_dataset(base_dir="data/cat_breeds_cleaned"):
    caption_count = 0
    breed_stats = {}

    for root, _, files in os.walk(base_dir):
        if root == base_dir:
            continue

        breed_name = os.path.basename(root)
        image_files = [f for f in files if f.lower().endswith('.jpg')]
        breed_stats[breed_name] = len(image_files)

        for file in image_files:
            caption = f"a photo of a {breed_name} cat"
            image_path = os.path.join(root, file)
            caption_path = os.path.splitext(image_path)[0] + ".txt"

            with open(caption_path, 'w', encoding='utf-8') as f:
                f.write(caption)

            caption_count += 1

    print(f"number of captions: {caption_count}")
    return breed_stats

breed_stats = create_captions_for_dataset()

number of captions: 8660


In [ ]:
def split_dataset_three_way(base_dir="data/cat_breeds_cleaned/cat_breeds_cleaned",
                             output_dir="data/cat_breeds_split",
                             train_ratio=0.7,
                             val_ratio=0.15,
                             seed=42):

    random.seed(seed)

    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    os.makedirs(f"{output_dir}/train", exist_ok=True)
    os.makedirs(f"{output_dir}/val", exist_ok=True)
    os.makedirs(f"{output_dir}/test", exist_ok=True)

    train_count = 0
    val_count = 0
    test_count = 0

    for breed_folder in sorted(os.listdir(base_dir)):
        breed_path = os.path.join(base_dir, breed_folder)

        if not os.path.isdir(breed_path):
            continue

        image_files = sorted([f for f in os.listdir(breed_path)
                              if f.lower().endswith('.jpg')])
        random.shuffle(image_files)

        train_end = int(len(image_files) * train_ratio)
        val_end = train_end + int(len(image_files) * val_ratio)

        train_images = image_files[:train_end]
        val_images = image_files[train_end:val_end]
        test_images = image_files[val_end:]

        for split, images in [("train", train_images), ("val", val_images), ("test", test_images)]:
            split_dir = os.path.join(output_dir, split, breed_folder)
            os.makedirs(split_dir, exist_ok=True)

            for img_file in images:
                shutil.copy2(os.path.join(breed_path, img_file),
                            os.path.join(split_dir, img_file))

                txt_file = img_file.replace('.jpg', '.txt')
                txt_src = os.path.join(breed_path, txt_file)
                if os.path.exists(txt_src):
                    shutil.copy2(txt_src, os.path.join(split_dir, txt_file))

        train_count += len(train_images)
        val_count += len(val_images)
        test_count += len(test_images)

        print(f"{breed_folder}: {len(train_images)} train, {len(val_images)} val, {len(test_images)} test")

    print(f"train: {train_count}")
    print(f"val: {val_count}")
    print(f"test: {test_count}")

    return train_count, val_count, test_count

train_count, val_count, test_count = split_dataset_three_way()

abyssinian: 130 train, 28 val, 29 test
american_bobtail: 94 train, 20 val, 21 test
american_curl: 93 train, 19 val, 21 test
american_shorthair: 129 train, 27 val, 29 test
american_wirehair: 123 train, 26 val, 27 test
balinese: 88 train, 19 val, 20 test
bengal: 104 train, 22 val, 23 test
birman: 90 train, 19 val, 20 test
british_shorthair: 118 train, 25 val, 27 test
burmese: 98 train, 21 val, 22 test
chartreux: 75 train, 16 val, 17 test
chausie: 130 train, 28 val, 29 test
cornish_rex: 136 train, 29 val, 30 test
cymric: 118 train, 25 val, 26 test
devon_rex: 95 train, 20 val, 21 test
donskoy: 115 train, 24 val, 26 test
egyptian_mau: 128 train, 27 val, 28 test
european_shorthair: 137 train, 29 val, 31 test
exotic_shorthair: 108 train, 23 val, 24 test
german_rex: 118 train, 25 val, 26 test
havana_brown: 43 train, 9 val, 10 test
japanese_bobtail: 118 train, 25 val, 27 test
karelian_bobtail: 75 train, 16 val, 17 test
khao_manee: 133 train, 28 val, 29 test
korat: 95 train, 20 val, 21 test
kore

**-> Eventuell ein Problem festgestellt:** Manche Ordner sind nach der Bereinigung sehr klein geworden (z.B. savannah). Evtl. führt das zu schlechteren Ergebnissen, was später zu testen wäre.

## **Trainings-Umgebung einrichten (UNet Finetuning)**

- Stable Diffusion 1.5 einrichten (SD 1.5, weil: Bilder haben sowieso eine kleine Auflösung/werden verpixelt sein, Training ist effizienter)
- Huggingface einrichten


In [ ]:
!pip install -U diffusers transformers accelerate

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### **Dataset erstellen**
Erstellt eine eigene Klasse, in der der Trainings- und Validierungsdatensatz umgewandelt wird und bereitet es für das Training vor
- Bilder & Texte werden umgewandelt

-> Bilder werden mit LANCZOS auf die gewünschte 512x512 skaliert, damit es zum Training passt, Pixelwerte werden zu 0-1 und wird später auf -1 bis 1 verschoben für Stabel Diffusion

-> Texte werden in Zahlen umgewandelt, kürzere Texte mit "padding" aufgefüllt und längere gekürzt -> erstellt Tokens


In [ ]:
class CatBreedDataset(Dataset):
    def __init__(self, data_dir, tokenizer, size=512):
        self.data_dir = Path(data_dir)
        self.tokenizer = tokenizer
        self.size = size

        self.image_paths = []
        self.captions = []

        for img_path in self.data_dir.rglob("*.jpg"):
            caption_path = img_path.with_suffix(".txt")
            if caption_path.exists():
                self.image_paths.append(img_path)
                with open(caption_path, 'r', encoding='utf-8') as f:
                    self.captions.append(f.read().strip())

        print(f"loaded dataset: {len(self.image_paths)} images")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        image = image.resize((self.size, self.size), Image.LANCZOS)
        image = np.array(image).astype(np.float32) / 255.0
        image = (image - 0.5) / 0.5
        image = torch.from_numpy(image).permute(2, 0, 1)

        caption = self.captions[idx]
        tokens = self.tokenizer(
            caption,
            padding="max_length",
            max_length=self.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt"
        )

        return {
            "pixel_values": image,
            "input_ids": tokens.input_ids[0],
        }

### **Einstellungen für das spätere Trainining**
- 512x512px Ausgabe
- 8 Bilder pro step
- 1 gradient_accumulation_steps, da 8 Bilder pro step Training stabil genug machen und ein höherer Wert das Training verlangsamen würde
- learning_rate 4e-5, Standard
- 24000 train steps, um ausreichend zu trainieren
- alle 3000 Schritte bisheriges Training speichern
- fp16 um GPU Speicher zu reduzieren (Qualitätsverlust sollte dadurch nicht dramatisch sein)

In [ ]:
config = {
    "model_id": "runwayml/stable-diffusion-v1-5",
    "train_data_dir": "data/cat_breeds_split/train",
    "output_dir": "cat-breed-model-hq",
    "resolution": 512,
    "train_batch_size": 8,
    "gradient_accumulation_steps": 1,
    "learning_rate": 4e-5,
    "max_train_steps": 24000,
    "save_steps": 3000,
    "mixed_precision": "fp16",
    "gradient_checkpointing": True,
    "enable_xformers": True,
}

### **Einstellungen für das feineres Trainining (Training von checkpoint-24k steps aus)**
- learning-rate auf 2e-5 reduziert, um feiner zu trainieren
- trainings steps auf 30000 erhöht


In [ ]:
config = {
    "model_id": "runwayml/stable-diffusion-v1-5",
    "train_data_dir": "data/cat_breeds_split/train",
    "output_dir": "cat-breed-model-hq",
    "resolution": 512,
    "train_batch_size": 8,
    "gradient_accumulation_steps": 1,
    "learning_rate": 2e-5,
    "max_train_steps": 30000,
    "save_steps": 3000,
    "mixed_precision": "fp16",
    "gradient_checkpointing": True,
    "enable_xformers": True,
}

### **Modell laden**
- legt accelerator fest (1 und fp16)
- Lädt alle Modellkomponenten (tokenizer, text_encoder, VAE, UNet, Noise Scheduler)
- VAE und text_encoder werden eingefroren, um nur UNet zu trainieren und finetuned auf meinen Katzen-Datensatz
- enable_gradient_checkpointing um mehr GPU-Speicher zu sparen

In [ ]:
accelerator = Accelerator(
    gradient_accumulation_steps=config["gradient_accumulation_steps"],
    mixed_precision=config["mixed_precision"],
)

tokenizer = CLIPTokenizer.from_pretrained(
    config["model_id"],
    subfolder="tokenizer"
)

text_encoder = CLIPTextModel.from_pretrained(
    config["model_id"],
    subfolder="text_encoder"
)

vae = AutoencoderKL.from_pretrained(
    config["model_id"],
    subfolder="vae"
)

unet = UNet2DConditionModel.from_pretrained(
    config["model_id"],
    subfolder="unet"
)

noise_scheduler = DDPMScheduler.from_pretrained(
    config["model_id"],
    subfolder="scheduler"
)

vae.requires_grad_(False)
text_encoder.requires_grad_(False)

if config["gradient_checkpointing"]:
    unet.enable_gradient_checkpointing()

vae = vae.to(accelerator.device, dtype=torch.float16)
text_encoder = text_encoder.to(accelerator.device, dtype=torch.float16)
unet = unet.to(accelerator.device)

tokenizer_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

### **Dataloader**
- Daten laden (train & val aus vorherigem split)
- train data Reihenfolge mischen und so anlegen, dass während ein Batch trainiert wird, das nächste vorbereitet wird
- val data nicht mischen & Vorbereitung nicht nötig

In [ ]:
train_dataset = CatBreedDataset(
    "data/cat_breeds_split/train/",
    tokenizer,
    size=config["resolution"]
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=config["train_batch_size"],
    shuffle=True,
    num_workers=2,
)

val_dataset = CatBreedDataset(
    "data/cat_breeds_split/val/",
    tokenizer,
    size=config["resolution"]
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=config["train_batch_size"],
    shuffle=False,
    num_workers=0,
)

print(f"train dataset: {len(train_dataset)} images")
print(f"val dataset: {len(val_dataset)} images")

loaded dataset: 6034 images
loaded dataset: 1274 images
train dataset: 6034 images
val dataset: 1274 images


### **AdamW Optimizer anlegen (Optimierungsalgorithmus)**
- entscheidet, wie UNet-Gewichte nach jedem step angepasst werden
- VAE und text_encoder sind nicht betroffen, da sie eingefroren sind
- betas glättet die Werte
- weight_decay verhindert, dass einzelne Gewichte zu groß werden (hilft gegen Overfitting)
- accelerator.prepare ersetzt alle Objekte durch GPU-optimierte Versionen
  (UNet für fp16, Optimizer mit GradScaler, beide DataLoader auf die GPU)

In [ ]:
optimizer = torch.optim.AdamW(
    unet.parameters(),
    lr=config["learning_rate"],
    betas=(0.9, 0.999),
    weight_decay=1e-2,
    eps=1e-08,
)

unet, optimizer, train_dataloader, val_dataloader = accelerator.prepare(
    unet, optimizer, train_dataloader, val_dataloader
)

### **Berechnung des Validierungs-Loss festlegen**
- schaltet UNet in den Evaluierungsmodus
- torch.no_grad: keine Gradienten berechnen, da nichts gelernt wird -> spart GPU-Speicher
- berechnet Durchnschnitt zwischen total_val_loss und num_batches
- wählt für jedes Bild im Batch einen zufälligen Zeitschritt und fügt später zufälliges Rauschen hinzu (bei timestep 100 z.B. wenig Rauschen, bei timestep 900 fast nur Rauschen)
- wandelt Token IDs zu einem Kontext Vektor um, damit UNet weiß, welches Bild es entrauschen soll
- UNet bekommt Bild und muss das Rauschen, das entfernt werden müsste, versuchen vorherzusagen
- MSE Loss vergleicht vorhergesagtes mit echtem Rauschen -> je kleiner desto besser
- gibt Durchschnitt über alle Val-Batches zurück und schaltet UNet zurück in den Trainingsmodus


In [ ]:
def compute_val_loss():
    unet.eval()
    total_val_loss = 0
    num_batches = 0

    with torch.no_grad():
        for batch in val_dataloader:
            latents = vae.encode(batch["pixel_values"].to(dtype=torch.float16)).latent_dist.sample()
            latents = latents * vae.config.scaling_factor

            noise = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps,
                (latents.shape[0],), device=latents.device
            ).long()

            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
            encoder_hidden_states = text_encoder(batch["input_ids"])[0]
            model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

            loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")
            total_val_loss += loss.item()
            num_batches += 1

    unet.train()
    return total_val_loss / num_batches if num_batches > 0 else 0

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

def plot_losses():
    fig, ax = plt.subplots(figsize=(12, 6))

    if len(losses) > 0:
        ax.plot(range(1, len(losses)+1), losses,
                label="Training Loss", color="steelblue", linewidth=2)

    if val_losses:
        ax.plot(val_steps, val_losses,
                label="Validation Loss", color="orange", linewidth=2)

    ax.set_title("Training and Validation Losses")
    ax.set_xlabel("Steps")
    ax.set_ylabel("Loss")
    ax.set_xlim(left=0)
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    plt.close()

## **Training**
- schaltet UNet in den Trainingsmodus
- Epochen auf max. 100 gestellt, wobei die Zahl hier relativ egal ist, Hauptsache sie liegt nicht unter den Epochen, die durch die Steps durchlaufen werden würden (bei 15k Steps ca. 20 Epochen)
- VAE komprimiert Bilder in den latenten Raum
- für jedes Bild im Batch wird ein zufälliger Zeitschritt gewählt und Rauschen hinzugefügt
- Text Encoder wandelt Tokens in Kontext-Vektor um
- UNet versucht das Rauschen vorherzusagen
- MSE Loss vergleicht vorhergesagtes mit echtem Rauschen
- backward() berechnet wie sehr jedes Gewicht zum Fehler beigetragen hat
- clip_grad_norm_ begrenzt Gradienten auf 1.0 -> verhindert Trainings-Instabilität
- optimizer.step() passt Gewichte an, optimizer.zero_grad() setzt Gradienten zurück
- alle 50 Steps: aktueller Loss und Durchschnitt der letzten 50 Steps wird ausgegeben
- alle 500 Steps: Val Loss berechnen und Plot aktualisieren
- alle 3000 Steps: Checkpoint speichern
- beide Schleifen (Batch & Epoche) stoppen wenn max_train_steps erreicht

In [ ]:
# training for 0 - 24000 steps!

losses = []
val_losses = []
val_steps = []

global_step = 0
progress_bar = tqdm(
    range(config["max_train_steps"]),
    disable=not accelerator.is_local_main_process,
)
progress_bar.set_description("Steps")

unet.train()

for epoch in range(100):
    for batch in train_dataloader:
        with accelerator.accumulate(unet):
            with torch.no_grad():
                latents = vae.encode(batch["pixel_values"].to(dtype=torch.float16)).latent_dist.sample()
                latents = latents * vae.config.scaling_factor

            noise = torch.randn_like(latents)
            bsz = latents.shape[0]

            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps, (bsz,),
                device=latents.device
            ).long()

            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            with torch.no_grad():
                encoder_hidden_states = text_encoder(batch["input_ids"])[0]

            model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
            loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")

            accelerator.backward(loss)
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(unet.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        if accelerator.sync_gradients:
            progress_bar.update(1)
            global_step += 1
            losses.append(loss.item())

            if global_step % 50 == 0:
                avg_loss = sum(losses[-50:]) / min(50, len(losses))
                print(f"Step {global_step}, Train: {loss.item():.4f}, Avg: {avg_loss:.4f}")

            if global_step % 500 == 0:
                val_loss = compute_val_loss()
                val_losses.append(val_loss)
                val_steps.append(global_step)
                print(f"val loss: {val_loss:.4f}")
                plot_losses()

            if global_step % config["save_steps"] == 0:
                save_path = os.path.join(config["output_dir"], f"checkpoint-{global_step}")
                accelerator.save_state(save_path)
                print(f"saved checkpoint: {save_path}")

        if global_step >= config["max_train_steps"]:
            break

    if global_step >= config["max_train_steps"]:
        break

print("training done!")

plot_losses()

### **Training nach 24000 steps -> Bis zu 30000 steps**
- Weiteres Finetuning, weitertrainieren mit neuen config-Einstellungen (learning-rate 2e-5), max_train_steps: 30000

In [ ]:
# training for 24000 - 30000 steps!

from accelerate.data_loader import skip_first_batches

losses = []
val_losses = []
val_steps = []

resume_from_checkpoint = "cat-breed-model-hq/checkpoint-24000"
starting_step = 24000

global_step = starting_step

progress_bar = tqdm(
    range(config["max_train_steps"]),
    initial=starting_step,
    disable=not accelerator.is_local_main_process,
)
progress_bar.set_description("Steps")

accelerator.load_state(resume_from_checkpoint)

for param_group in optimizer.param_groups:
    param_group['lr'] = config["learning_rate"]
print(f"Learning rate set to: {config['learning_rate']}")

unet.train()

steps_per_epoch = len(train_dataloader) // config["gradient_accumulation_steps"]
epochs_completed = starting_step // steps_per_epoch
steps_in_current_epoch = starting_step % steps_per_epoch

for epoch in range(epochs_completed, 100):

    if epoch == epochs_completed and steps_in_current_epoch > 0:
        active_dataloader = skip_first_batches(train_dataloader, steps_in_current_epoch)
        print(f"Epoch {epoch}: Skipping first {steps_in_current_epoch} batches")
    else:
        active_dataloader = train_dataloader

    for batch in active_dataloader:
        with accelerator.accumulate(unet):
            with torch.no_grad():
                latents = vae.encode(batch["pixel_values"].to(dtype=torch.float16)).latent_dist.sample()
                latents = latents * vae.config.scaling_factor

            noise = torch.randn_like(latents)
            bsz = latents.shape[0]

            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps, (bsz,),
                device=latents.device
            ).long()

            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            with torch.no_grad():
                encoder_hidden_states = text_encoder(batch["input_ids"])[0]

            model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
            loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")

            accelerator.backward(loss)
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(unet.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        if accelerator.sync_gradients:
            progress_bar.update(1)
            global_step += 1
            losses.append(loss.item())

            if global_step % 50 == 0:
                avg_loss = sum(losses[-50:]) / min(50, len(losses))
                print(f"Step {global_step}, Train: {loss.item():.4f}, Avg: {avg_loss:.4f}")

            if global_step % 500 == 0:
                val_loss = compute_val_loss()
                val_losses.append(val_loss)
                val_steps.append(global_step)
                print(f"val loss: {val_loss:.4f}")
                plot_losses()

            if global_step % config["save_steps"] == 0:
                save_path = os.path.join(config["output_dir"], f"checkpoint-{global_step}")
                accelerator.save_state(save_path)
                print(f"saved checkpoint: {save_path}")

        if global_step >= config["max_train_steps"]:
            break

    if global_step >= config["max_train_steps"]:
        break

print("training done!")

plot_losses()

## **Datenanalyse**
- von 0 bis 15000 steps

In [ ]:
def plot_train_val_curves(smooth_window=50):
    fig, ax = plt.subplots(figsize=(12, 8))

    if len(losses) > 0:
        if smooth_window > 1 and len(losses) >= smooth_window:
            train_smoothed = np.convolve(losses, np.ones(smooth_window)/smooth_window, mode='valid')
            x_train = range(smooth_window, len(losses)+1)
            ax.plot(x_train, train_smoothed,
                   label="train", color="#1f77b4", linewidth=2.5)
        else:
            ax.plot(range(1, len(losses)+1), losses,
                   label="train", color="#1f77b4", linewidth=2.5)

    if len(val_losses) > 0:
        ax.plot(val_steps, val_losses,
               label="validation", color="#ff7f0e", linewidth=2.5)

    ax.set_xlabel("steps", fontsize=14)
    ax.set_ylabel("loss", fontsize=14)
    ax.set_title("train-val loss", fontsize=16, pad=20)
    ax.legend(fontsize=12, frameon=True, fancybox=True)
    ax.grid(False)
    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)

    plt.tight_layout()
    plt.show()
    plt.close()

plot_train_val_curves(smooth_window=100)

### **Datenanalyse**
- von 15000/24000 bis 30000 steps

In [ ]:
def plot_train_val_curves(smooth_window=50):
    fig, ax = plt.subplots(figsize=(12, 8))

    if len(losses) > 0:
        x_all = range(starting_step + 1, starting_step + len(losses) + 1)

        if smooth_window > 1 and len(losses) >= smooth_window:
            train_smoothed = np.convolve(losses, np.ones(smooth_window)/smooth_window, mode='valid')
            x_train = range(starting_step + smooth_window, starting_step + len(losses) + 1)
            ax.plot(x_train, train_smoothed,
                   label="train", color="#1f77b4", linewidth=2.5)
        else:
            ax.plot(x_all, losses,
                   label="train", color="#1f77b4", linewidth=2.5)

    if len(val_losses) > 0:
        ax.plot(val_steps, val_losses,
               label="validation", color="#ff7f0e", linewidth=2.5)

    ax.set_xlabel("steps", fontsize=14)
    ax.set_ylabel("loss", fontsize=14)
    ax.set_title("train-val loss", fontsize=16, pad=20)
    ax.legend(fontsize=12, frameon=True, fancybox=True)
    ax.grid(False)
    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)
    ax.set_xlim(left=starting_step)

    plt.tight_layout()
    plt.show()
    plt.close()

plot_train_val_curves(smooth_window=100)

## **Modell erstellen**
- Erstellt Modell aus Checkpoint

In [ ]:
from safetensors.torch import load_file
from diffusers import StableDiffusionPipeline, UNet2DConditionModel

unet = UNet2DConditionModel.from_pretrained(
    config["model_id"], subfolder="unet", torch_dtype=torch.float16
)
state_dict = load_file("cat-breed-model-hq/checkpoint-27000/model.safetensors", device="cpu")
unet.load_state_dict(state_dict)

final_pipeline = StableDiffusionPipeline.from_pretrained(
    config["model_id"],
    unet=unet,
    torch_dtype=torch.float16,
)
final_pipeline.save_pretrained(f"{config['output_dir']}/final_model_27k")

del final_pipeline, unet
torch.cuda.empty_cache()

## **Bildgenerierung**
- Verschiedene Cases für die Bildgenerierung (zwecks Umfrage)
- num_inference_steps = 50, guidance_scale = 7.5 (Standardwerte aber trotzdem ergänzt, um es zu erzwingen)
- pipe_standard.safety_checker = None -> wegen Sphynx, wurde sonst als NSFW gekennzeichnet und als schwarzes Bild generiert
- Immer die gleichen Seeds, damit verschiedene Checkpoint-Modelle/Prompts gut miteinander verglichen werden können

**Verschiedene Modelle testen**
- Es wurden erst verschiedene Modelle getestet (checkpoint 3000, 9000, 15000, 24000, 27000 und 30000)

**Verschiedene Rassen testen**
- Es wurden alle Rassen außer Bengal, Maine Coon & Sphynx getestet (da bereits getestet)

**Bildgenerierung für Umfragen**
- Verschiedene Prompts & diese Prompts mehrfach generieren, damit Ergebnisse vergleichbar sind

**Plan für Umfrage**
- Immer die ersten drei Bilder nehmen, KEIN cherry picking der Bilder
- Nicht nennen, welches Modell welches ist, um "bias" zu vermeiden
- Erst einzelne Katzen zeigen, dann Kreuzungen NUR aus dem 1. Prompt, am Ende Prompts vergleichen

In [ ]:
# to test a checkpoint-model & for Google survey!

import torch
from diffusers import StableDiffusionPipeline
from IPython.display import display

prompts = [
    # one breed only
    "a photo of a sphynx cat",
    "a photo of a maine_coon cat",
    "a photo of a bengal cat",

    # cross-breeds
    "a photo of a sphynx mixed with a maine_coon cat",
    "a photo of a sphynx mixed with a bengal cat",
    "a photo of a maine_coon mixed with a bengal cat",
]

pipe_trained = StableDiffusionPipeline.from_pretrained(
    f"{config['output_dir']}/final_model_27k",
    torch_dtype=torch.float16,
).to("cuda")
pipe_trained.safety_checker = None

seeds = [42, 38, 212, 342, 678, 982]
for prompt in prompts:
    print(f"prompt: {prompt}")

    generator = torch.Generator("cuda").manual_seed(seeds)
    img_trained = pipe_trained(prompt, generator=generator, num_inference_steps = 50,
        guidance_scale = 7.5).images[0]

    display(img_trained)

In [ ]:
# to test every other bread (except sphynx, bengal, maine coon)

import torch
from diffusers import StableDiffusionPipeline
from IPython.display import display

prompts = [
    "a photo of a abyssinian cat",
    "a photo of a american_bobtail cat",
    "a photo of a american_curl cat",
    "a photo of a american_shorthair cat",
    "a photo of a american_wirehair cat",
    "a photo of a balinese cat",
    "a photo of a birman cat",
    "a photo of a british_shorthair cat",
    "a photo of a burmese cat",
    "a photo of a chartreux cat",
    "a photo of a chausie cat",
    "a photo of a cornish_rex cat",
    "a photo of a cymric cat",
    "a photo of a devon_rex cat",
    "a photo of a donskoy cat",
    "a photo of a egyptian_mau cat",
    "a photo of a european_shorthair cat",
    "a photo of a exotic_shorthair cat",
    "a photo of a german_rex cat",
    "a photo of a havana_brown cat",
    "a photo of a japanese_bobtail cat",
    "a photo of a karelian_bobtail cat",
    "a photo of a khao_manee cat",
    "a photo of a korat cat",
    "a photo of a korean_bobtail cat",
    "a photo of a kurilian_bobtail cat",
    "a photo of a laperm cat",
    "a photo of a lykoi cat",
    "a photo of a manx cat",
    "a photo of a mekong_bobtail cat",
    "a photo of a munchkin cat",
    "a photo of a nebelung cat",
    "a photo of a norwegian_forest_cat cat",
    "a photo of a ocicat cat",
    "a photo of a oregon_rex cat",
    "a photo of a oriental_shorthair cat",
    "a photo of a persian cat",
    "a photo of a peterbald cat",
    "a photo of a pixie_bob cat",
    "a photo of a ragamuffin cat",
    "a photo of a ragdoll cat",
    "a photo of a russian_blue cat",
    "a photo of a savannah cat",
    "a photo of a scottish_fold cat",
    "a photo of a selkirk_rex cat",
    "a photo of a siamese cat",
    "a photo of a siberian cat",
    "a photo of a singapura cat",
    "a photo of a sokoke cat",
    "a photo of a tonkinese cat",
    "a photo of a toyger cat",
    "a photo of a turkish_angora cat",
    "a photo of a turkish_van cat",
    "a photo of a ukrainian_levkoy cat",
    "a photo of a ural_rex cat",
    "a photo of a vankedisi cat",
]

pipe_trained = StableDiffusionPipeline.from_pretrained(
    f"{config['output_dir']}/final_model_24k",
    torch_dtype=torch.float16,
).to("cuda")
pipe_trained.safety_checker = None

seeds = [212, 342]

for prompt in prompts:
    print(f"prompt: {prompt}")

    for seed in seeds:
        generator = torch.Generator("cuda").manual_seed(seed)
        img_trained = pipe_trained(prompt, generator=generator, num_inference_steps=50,
            guidance_scale=7.5).images[0]

        display(img_trained)

In [ ]:
#to test prompts for Google survey! (section "Prompts vergleichen & bewerten")

import torch
from diffusers import StableDiffusionPipeline
from IPython.display import display

prompts = [
    #prompt 1 sphynx x maine-coon
    "a photo of a sphynx mixed with a maine_coon cat",
    "a photo of a cat that is half sphynx cat and half maine_coon cat",
    "a photo of a sphynx cat x maine_coon cat",
    "a photo of a sphynx cat + maine_coon cat",

    #prompt 2 sphynx x bengal
    "a photo of a sphynx mixed with a bengal cat",
    "a photo of a cat that is half sphynx cat and half bengal cat",
    "a photo of a sphynx cat x bengal cat",
    "a photo of a sphynx cat + bengal cat",

    #prompt 2 maine-coon x bengal
    "a photo of a maine_coon mixed with a bengal cat",
    "a photo of a cat that is half maine_coon cat and half bengal cat",
    "a photo of a maine_coon cat x bengal cat",
    "a photo of a maine_coon cat + bengal cat",
]

pipe_standard = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
).to("cuda")
pipe_standard.safety_checker = None

pipe_trained = StableDiffusionPipeline.from_pretrained(
    f"{config['output_dir']}/final_model_24k",
    torch_dtype=torch.float16,
).to("cuda")
pipe_trained.safety_checker = None

seeds = [48, 41, 40, 56, 68, 32]

for prompt in prompts:
    print(f"prompt: {prompt}")

    for seed in seeds:
        print(f"seed: {seed}")

        generator = torch.Generator("cuda").manual_seed(seed)
        img_standard = pipe_standard(prompt, generator=generator, num_inference_steps = 50,
        guidance_scale = 7.5).images[0]

        generator = torch.Generator("cuda").manual_seed(seed)
        img_trained = pipe_trained(prompt, generator=generator, num_inference_steps = 50,
        guidance_scale = 7.5).images[0]

        print("standard SD 1.5:")
        display(img_standard)

        print("model 24k:")
        display(img_trained)

## **Fazit**

- Da die Ergebnisse mit einer Google Umfrage evaluiert worden sind, hat sich herausgestellt, dass der vorherige Split des Datensatzes etwas ungünstig war, denn: 15% des Datensatzes sind Testdaten, die später für die Evaluierung genutzt werden sollten, letztlich aber nicht genutzt worden sind. Das sorgt dafür, dass der bereits relativ kleine Datensatz nach der Bereinigung noch kleiner wurde. Eventuell wäre mit den 15% mehr das Training etwas besser gelaufen.

- Die besten Ergebnisse konnten mit 24000 Steps erreicht werden, bei 30000 Steps mit feinerer Learning Rate waren die Ergebnisse kaum besser & es kaum eher zu starkem Overfitting